# PoseMamba: 3D pose from a video (Colab)

Upload a video of a person, get back a video with the 2D skeleton on the footage and a 3D skeleton beside it.

**Two cells to run each session:** **Setup**, then **Run** (and the last cell to download the result).

- **First time ever:** Setup takes about 20 minutes because it compiles GPU kernels, and it saves them to your Google Drive. It will ask permission to use Drive.
- **Every later session:** Setup restores the saved kernels and takes about 3 minutes.
- **Within one session:** after Setup, run **Run** as many times as you like with new videos.

Before you start: Runtime > Change runtime type > **T4 GPU**.

## Setup (run once per session)

In [ ]:
import os, sys, subprocess, glob, shutil, json, torch

assert torch.cuda.is_available(), "No GPU found. Runtime > Change runtime type > T4 GPU, then restart."
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__, "| CUDA", torch.version.cuda)

ROOT  = "/content/PoseMamba"
KDIR  = ROOT + "/kernels/selective_scan"
CACHE = "/content/drive/MyDrive/posemamba_cache"   # where compiled kernels are kept between sessions

# --- 1. project code and Python packages -----------------------------------------
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/nankingjing/PoseMamba.git", ROOT], check=True)
os.chdir(ROOT)
!pip install -q -r requirements.txt gdown

# --- 2. patches for modern CUDA / PyTorch ----------------------------------------
# CUDA 13 removed cub::LaneId and cub::CTA_SYNC; the repo builds only 'oflex' but the model also needs 'core'
!sed -i 's/cub::LaneId()/(threadIdx.x \& 31)/; s/cub::CTA_SYNC()/__syncthreads()/' kernels/selective_scan/csrc/selective_scan/reverse_scan.cuh
!sed -i 's/^MODES = \["oflex"\]/MODES = ["core", "oflex"]/' kernels/selective_scan/setup.py
# PyTorch 2.6+ refuses older checkpoints unless told to; these weights come from the project authors' own repo
src = open("vis.py").read()
if "weights_only" not in src:
    open("vis.py", "w").write(src.replace("map_location=lambda storage, loc: storage)",
                                          "map_location=lambda storage, loc: storage, weights_only=False)"))

# --- 3. GPU kernels: restore from Drive if possible, otherwise compile and save ---
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(CACHE, exist_ok=True)

stamp = {"python": sys.version.split()[0], "torch": torch.__version__, "cuda": torch.version.cuda,
         "gpu_cc": list(torch.cuda.get_device_capability(0))}

def kernels_ok():
    # test in a fresh process so a broken file can't poison this session
    code = "import torch, selective_scan_cuda_core, selective_scan_cuda_oflex"
    r = subprocess.run([sys.executable, "-c", code], env=dict(os.environ, PYTHONPATH=KDIR), capture_output=True)
    return r.returncode == 0

stamp_file = CACHE + "/stamp.json"
if not kernels_ok() and os.path.exists(stamp_file) and json.load(open(stamp_file)) == stamp:
    for f in glob.glob(CACHE + "/*.so"):
        shutil.copy(f, KDIR)
    print("restored kernels from Drive")

if not kernels_ok():
    print("no usable saved kernels (first run, or Colab's software changed). Compiling, about 10 minutes...")
    os.chdir(KDIR)
    r = subprocess.run("python setup.py build_ext --inplace > /content/build.log 2>&1", shell=True)
    if r.returncode != 0:
        !grep -nE " error|Error:" /content/build.log | head -20
        raise SystemExit("Kernel build failed, see the errors above.")
    for f in glob.glob(KDIR + "/*.so"):
        shutil.copy(f, CACHE)
    json.dump(stamp, open(stamp_file, "w"))
    print("compiled and saved kernels to Drive for next time")
    os.chdir(ROOT)

assert kernels_ok(), "Kernels still do not load."
sys.path.insert(0, KDIR)
os.environ["PYTHONPATH"] = KDIR
os.chdir(ROOT)
print("kernels OK")

# --- 4. model weights and detector files -----------------------------------------
from huggingface_hub import hf_hub_download
os.makedirs("checkpoint/pose3d/PoseMamba_L", exist_ok=True)
w = hf_hub_download("nankingjings/PoseMamba-weights", "PoseMamba_L.bin")
shutil.copy(w, "checkpoint/pose3d/PoseMamba_L/best_epoch.bin")
shutil.copy("configs/pose3d/PoseMamba_train_h36m_L.yaml", "checkpoint/pose3d/PoseMamba_L/config.yaml")

# The demo/ folder (2D person detector code, its weights, sample videos; ~1.1 GB) is not in the git repo.
# It comes from the Google Drive file linked in the PoseMamba README.
if not os.path.isdir("demo/lib/checkpoint"):
    !gdown 1hbK1HDz1nMTGYcczOC5r33Mk8nAtLZCr -O /content/demo.tar
    !tar -xf /content/demo.tar -C /content/PoseMamba

print("\nSetup complete. Run the next cell.")

## Run it on a video

`USE_SAMPLE = True` uses the bundled clip. Set it to `False` to upload your own
(one person, clear view; short clips are faster, drawing takes about 1 second per frame).

In [ ]:
USE_SAMPLE = True   # True = bundled sample clip, False = upload your own

os.chdir(ROOT)
os.environ["PYTHONPATH"] = KDIR
if USE_SAMPLE:
    video_name = "tj_cut.mp4"
else:
    from google.colab import files
    up = files.upload()
    video_name = list(up.keys())[0]
    shutil.move(video_name, "demo/video/" + video_name)

stem = os.path.splitext(video_name)[0]
!python vis.py --video {video_name} --gpu 0 2>&1 | grep -v -e FutureWarning -e "@torch"

out = f"{ROOT}/demo/output/{stem}/{stem}.mp4"
print("\nOutput video:", out, "| exists:", os.path.exists(out))

In [ ]:
# Download the result to your computer
from google.colab import files
files.download(out)